# YOLO CIFAR10 Image Source: Teacher → Pruned Student → Logits Distillation (Classification)

This notebook uses CIFAR10 images as an input source while running a YOLOv8n **classification** pruning + logits distillation flow:
1. Load a CIFAR10 fine-tuned YOLOv8n classification teacher checkpoint.
2. Build a `MaseGraph` from the same checkpoint and run unstructured pruning to create the student.
3. Distill teacher logits into the pruned student using CIFAR10 mini-batches (images + labels).

In [1]:
from pathlib import Path

# Prefer a fine-tuned CIFAR10 teacher checkpoint produced by `cw/yolov8n_cifar.ipynb`.
teacher_runs_roots = [
    Path("./data/cifar10_yolov8n_cls/runs"),
    Path("./cw/data/cifar10_yolov8n_cls/runs"),
]

teacher_runs_root = next((path for path in teacher_runs_roots if path.exists()), teacher_runs_roots[0])
teacher_default_checkpoint = teacher_runs_root / "yolov8n_cls_cifar10_finetune4" / "weights" / "best.pt"

if teacher_default_checkpoint.exists():
    cls_model_checkpoint = str(teacher_default_checkpoint)
else:
    candidate_best = sorted(teacher_runs_root.glob("*/weights/best.pt"))
    cls_model_checkpoint = str(candidate_best[-1]) if candidate_best else "yolov8n-cls.pt"

cls_device = "cuda"
cifar_root = "./data"

# Match the CIFAR10 fine-tuning resolution.
image_size = 32
cifar_batch_size = 16
cifar_train_subset_size = 2048
cifar_val_subset_size = 512

cifar_prune_sparsity = 0.3
cifar_kd_alpha = 0.8
cifar_kd_temperature = 2.0
cifar_kd_steps = 200
lr = 1e-4
seed = 42

print(f"Teacher runs root: {teacher_runs_root}")
print(f"Teacher checkpoint: {cls_model_checkpoint}")

Teacher runs root: data/cifar10_yolov8n_cls/runs
Teacher checkpoint: data/cifar10_yolov8n_cls/runs/yolov8n_cls_cifar10_finetune4/weights/best.pt


## Imports and setup

In [ ]:
import copy
import random
import sys
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from ultralytics import YOLO

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate repository root containing src/")

repo_root = find_repo_root(Path.cwd().resolve())
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from chop import MaseGraph
import chop.passes as passes
from chop.models.yolo import get_yolo_classification_model
from chop.models.yolo.yolov8 import patch_yolo

from mase_kd.core.losses import DistillationLossConfig
from mase_kd.vision.yolo_kd import YOLOLogitsDistiller

print(f"Repo root: {repo_root}")
print(f"Using src path: {src_path}")

Repo root: /workspace/mase-kd
Using src path: /workspace/mase-kd/src


ERROR! Session/line number was not unique in database. History logging moved to new session 84


In [3]:
if cls_device == "cuda" and not torch.cuda.is_available():
    cls_device = "cpu"

torch.manual_seed(seed)
random.seed(seed)

print(f"Using device: {cls_device}")

Using device: cuda


## CIFAR10 image dataset and dataloaders

In [4]:
cifar_mean = (0.4914, 0.4822, 0.4465)
cifar_std = (0.2470, 0.2435, 0.2616)

cifar_transform_train = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(cifar_mean, cifar_std),
])
cifar_transform_eval = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(cifar_mean, cifar_std),
])

train_full = datasets.CIFAR10(root=cifar_root, train=True, transform=cifar_transform_train, download=True)
val_full = datasets.CIFAR10(root=cifar_root, train=False, transform=cifar_transform_eval, download=True)

subset_rng = torch.Generator().manual_seed(seed)
train_indices = torch.randperm(len(train_full), generator=subset_rng)[:cifar_train_subset_size]
val_indices = torch.randperm(len(val_full), generator=subset_rng)[:cifar_val_subset_size]

train_ds = Subset(train_full, train_indices.tolist())
val_ds = Subset(val_full, val_indices.tolist())

# drop_last=True keeps batch size fixed for the FX-specialized pruned student graph.
train_loader = DataLoader(
    train_ds,
    batch_size=cifar_batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)
val_loader = DataLoader(
    val_ds,
    batch_size=cifar_batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)

print(f"Train subset: {len(train_ds)} | Val subset: {len(val_ds)}")

Train subset: 2048 | Val subset: 512


## Load teacher model from fine-tuned YOLOv8n classification checkpoint

In [5]:
ultra_teacher = YOLO(cls_model_checkpoint)
teacher_cls_model = ultra_teacher.model.to(cls_device)
teacher_cls_model.eval()

print(f"Loaded teacher checkpoint: {cls_model_checkpoint}")

Loaded teacher checkpoint: data/cifar10_yolov8n_cls/runs/yolov8n_cls_cifar10_finetune4/weights/best.pt


## Build MASE graph and prune to create student

In [6]:
trace_input_cls = torch.randn(cifar_batch_size, 3, image_size, image_size)

# Build student as a pruned copy of the loaded CIFAR10-trained teacher model.
student_seed_cls_model = copy.deepcopy(teacher_cls_model).to(cls_device)
student_seed_cls_model = patch_yolo(student_seed_cls_model)
student_seed_cls_model.train()

mg_cls = MaseGraph(student_seed_cls_model, cf_args={"x": trace_input_cls})
mg_cls, _ = passes.init_metadata_analysis_pass(mg_cls)

placeholder_names_cls = [
    node.name for node in mg_cls.fx_graph.nodes if node.op == "placeholder"
]
dummy_in_cls = {name: trace_input_cls for name in placeholder_names_cls}

mg_cls, _ = passes.add_common_metadata_analysis_pass(
    mg_cls,
    pass_args={
        "dummy_in": dummy_in_cls,
        "add_value": True,
    },
)

pruning_config_cls = {
    "weight": {
        "sparsity": cifar_prune_sparsity,
        "method": "l1-norm",
        "scope": "local",
        "granularity": "elementwise",
    },
    "activation": {
        "sparsity": cifar_prune_sparsity,
        "method": "l1-norm",
        "scope": "local",
        "granularity": "elementwise",
    },
}

mg_cls, _ = passes.prune_transform_pass(mg_cls, pass_args=pruning_config_cls)
student_cls_model = mg_cls.model.to(cls_device)

print("Pruning complete; using pruned teacher copy as student.")

Unexpected exception formatting exception. Falling back to standard exception
Unexpected exception formatting exception. Falling back to standard exception
Unexpected exception formatting exception. Falling back to standard exception


## Distill teacher into pruned student on CIFAR10 images + labels

In [9]:
kd_config_cls = DistillationLossConfig(
    alpha=cifar_kd_alpha,
    temperature=cifar_kd_temperature,
)
distiller_cls = YOLOLogitsDistiller(
    teacher=teacher_cls_model,
    student=student_cls_model,
    kd_config=kd_config_cls,
    device=cls_device,
)

pruned_no_kd_model = copy.deepcopy(student_cls_model).to(cls_device)
pruned_no_kd_model.eval()

optimizer = torch.optim.Adam(student_cls_model.parameters(), lr=lr)
loss_history = []

def _extract_logits_with_batch(output, batch_size):
    if isinstance(output, torch.Tensor):
        if output.ndim >= 2 and output.shape[0] == batch_size:
            return output.reshape(batch_size, -1)
        if output.ndim >= 2 and output.shape[0] == 1 and batch_size > 1:
            return output.reshape(1, -1).expand(batch_size, -1)
        return None

    if isinstance(output, dict):
        for key in sorted(output.keys()):
            value = output[key]
            if isinstance(value, (torch.Tensor, dict, list, tuple)):
                logits = _extract_logits_with_batch(value, batch_size)
                if logits is not None:
                    return logits
        return None

    if isinstance(output, (list, tuple)):
        for item in output:
            if isinstance(item, (torch.Tensor, dict, list, tuple)):
                logits = _extract_logits_with_batch(item, batch_size)
                if logits is not None:
                    return logits
        return None

    return None

def classification_hard_loss(student_output, batch):
    targets = batch["targets"]
    logits = _extract_logits_with_batch(student_output, targets.shape[0])
    if logits is None:
        raise ValueError(
            "Could not find student logits matching target batch size for hard CE loss. "
            "Check trace_input_cls batch size vs DataLoader batch size."
        )

    max_label = int(targets.max().item())
    if logits.shape[1] <= max_label:
        raise ValueError(
            f"Student logits class dim ({logits.shape[1]}) <= max target ({max_label})."
        )

    return torch.nn.functional.cross_entropy(logits, targets)

train_iter = iter(train_loader)
for step in range(1, cifar_kd_steps + 1):
    try:
        images, labels = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        images, labels = next(train_iter)

    batch = {
        "images": images.to(cls_device),
        "targets": labels.to(cls_device),
    }

    if step == 1:
        with torch.no_grad():
            sample_out = student_cls_model(batch["images"])
        sample_logits = _extract_logits_with_batch(sample_out, batch["targets"].shape[0])
        if sample_logits is None:
            print("Step 1 debug: could not extract batch-aligned logits from student output.")
        else:
            print(
                f"Step 1 debug: extracted logits shape={tuple(sample_logits.shape)} "
                f"for targets shape={tuple(batch['targets'].shape)}"
            )

    output = distiller_cls.train_step(
        batch=batch,
        optimizer=optimizer,
        task_loss_fn=classification_hard_loss,
    )
    loss_history.append(output.total_loss)

    if step == 1 or step % 10 == 0 or step == cifar_kd_steps:
        print(
            f"Step {step:03d} | total={output.total_loss:.6f} | "
            f"hard={output.hard_loss:.6f} | soft={output.soft_loss:.6f}"
        )

Step 1 debug: extracted logits shape=(16, 1000) for targets shape=(16,)
Step 001 | total=4.311590 | hard=8.605115 | soft=3.238209
Step 010 | total=3.943819 | hard=7.532975 | soft=3.046530
Step 020 | total=3.258782 | hard=6.543360 | soft=2.437637
Step 030 | total=3.048974 | hard=6.122315 | soft=2.280639
Step 040 | total=2.911314 | hard=5.751457 | soft=2.201279
Step 050 | total=2.984031 | hard=5.434232 | soft=2.371480
Step 060 | total=2.852883 | hard=5.120985 | soft=2.285857
Step 070 | total=2.732364 | hard=4.307557 | soft=2.338566
Step 080 | total=2.727878 | hard=3.732442 | soft=2.476737
Step 090 | total=2.475761 | hard=3.712992 | soft=2.166453
Step 100 | total=3.018320 | hard=3.878933 | soft=2.803167
Step 110 | total=2.723156 | hard=3.967105 | soft=2.412169
Step 120 | total=2.660424 | hard=3.753831 | soft=2.387073
Step 130 | total=2.513183 | hard=3.693435 | soft=2.218120
Step 140 | total=2.608207 | hard=3.590940 | soft=2.362523
Step 150 | total=2.674104 | hard=3.445208 | soft=2.481328


## Validation performance (teacher vs student)

In [11]:
import time

def _extract_logits_with_batch(output, batch_size):
    if isinstance(output, torch.Tensor):
        if output.ndim >= 2 and output.shape[0] == batch_size:
            return output.reshape(batch_size, -1)
        if output.ndim >= 2 and output.shape[0] == 1 and batch_size > 1:
            return output.reshape(1, -1).expand(batch_size, -1)
        return None

    if isinstance(output, dict):
        for key in sorted(output.keys()):
            value = output[key]
            if isinstance(value, (torch.Tensor, dict, list, tuple)):
                logits = _extract_logits_with_batch(value, batch_size)
                if logits is not None:
                    return logits
        return None

    if isinstance(output, (list, tuple)):
        for item in output:
            if isinstance(item, (torch.Tensor, dict, list, tuple)):
                logits = _extract_logits_with_batch(item, batch_size)
                if logits is not None:
                    return logits
        return None

    return None

def _align_logits_for_kd(student_logits, teacher_logits):
    width = min(student_logits.shape[1], teacher_logits.shape[1])
    if width == 0:
        return None, None
    return student_logits[:, :width], teacher_logits[:, :width]

@torch.no_grad()
def evaluate_model_on_cifar10_val(model, loader, device, max_batches=8):
    model.eval()
    batches = 0
    samples = 0
    sum_abs_logit = 0.0
    sum_sq_logit = 0.0
    total_forward_ms = 0.0
    correct_top1 = 0
    acc_samples = 0

    for batch_idx, (images, labels) in enumerate(loader):
        if batch_idx >= max_batches:
            break

        images = images.to(device)
        labels = labels.to(device)
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        outputs = model(images)
        if device == "cuda":
            torch.cuda.synchronize()
        t1 = time.perf_counter()

        logits = _extract_logits_with_batch(outputs, images.shape[0])
        if logits is None or logits.numel() == 0:
            continue

        total_forward_ms += (t1 - t0) * 1000.0
        sum_abs_logit += logits.abs().mean().item()
        sum_sq_logit += logits.pow(2).mean().item()
        batches += 1
        samples += images.shape[0]

        max_label = int(labels.max().item())
        if logits.shape[1] > max_label:
            preds = logits.argmax(dim=1)
            correct_top1 += int((preds == labels).sum().item())
            acc_samples += int(labels.shape[0])

    return {
        "batches": batches,
        "samples": samples,
        "avg_forward_ms_per_batch": total_forward_ms / max(batches, 1),
        "avg_abs_logit": sum_abs_logit / max(batches, 1),
        "avg_logit_l2": (sum_sq_logit / max(batches, 1)) ** 0.5,
        "top1_acc": correct_top1 / max(acc_samples, 1),
        "acc_samples": acc_samples,
    }

@torch.no_grad()
def evaluate_validation_losses(teacher, student, loader, device, temperature=2.0, max_batches=8):
    teacher.eval()
    student.eval()
    total_teacher_loss = 0.0
    total_student_loss = 0.0
    total_kd_loss = 0.0
    used_batches = 0

    for batch_idx, (images, labels) in enumerate(loader):
        if batch_idx >= max_batches:
            break

        images = images.to(device)
        labels = labels.to(device)
        teacher_outputs = teacher(images)
        student_outputs = student(images)

        teacher_logits = _extract_logits_with_batch(teacher_outputs, labels.shape[0])
        student_logits = _extract_logits_with_batch(student_outputs, labels.shape[0])
        if teacher_logits is None or student_logits is None:
            continue

        student_logits, teacher_logits = _align_logits_for_kd(student_logits, teacher_logits)
        if student_logits is None or teacher_logits is None:
            continue

        max_label = int(labels.max().item())
        if student_logits.shape[1] <= max_label or teacher_logits.shape[1] <= max_label:
            continue

        teacher_loss = torch.nn.functional.cross_entropy(teacher_logits, labels)
        student_loss = torch.nn.functional.cross_entropy(student_logits, labels)

        student_log_probs = torch.nn.functional.log_softmax(student_logits / temperature, dim=1)
        teacher_probs = torch.nn.functional.softmax(teacher_logits / temperature, dim=1)
        kd_loss = torch.nn.functional.kl_div(
            student_log_probs, teacher_probs, reduction="batchmean"
        ) * (temperature ** 2)

        total_teacher_loss += teacher_loss.item()
        total_student_loss += student_loss.item()
        total_kd_loss += kd_loss.item()
        used_batches += 1

    return (
        total_teacher_loss / max(used_batches, 1),
        total_student_loss / max(used_batches, 1),
        total_kd_loss / max(used_batches, 1),
        used_batches,
    )

if "loss_history" in globals() and len(loss_history) > 0:
    print(f"Recorded {len(loss_history)} KD steps")
    print(f"First loss: {loss_history[0]:.6f}")
    print(f"Last loss:  {loss_history[-1]:.6f}")
else:
    print("No training loss history found in current kernel session.")

teacher_metrics = evaluate_model_on_cifar10_val(
    teacher_cls_model, val_loader, cls_device, max_batches=8
)
student_metrics = evaluate_model_on_cifar10_val(
    student_cls_model, val_loader, cls_device, max_batches=8
)
teacher_val_loss, student_val_loss, val_kd_loss, loss_batches = evaluate_validation_losses(
    teacher_cls_model, student_cls_model, val_loader, cls_device,
    temperature=cifar_kd_temperature, max_batches=8
)

pruned_no_kd_val_loss = None
pruned_no_kd_metrics = None
if "pruned_no_kd_model" in globals():
    pruned_no_kd_metrics = evaluate_model_on_cifar10_val(
        pruned_no_kd_model, val_loader, cls_device, max_batches=8
    )
    _, pruned_no_kd_val_loss, _, _ = evaluate_validation_losses(
        teacher_cls_model, pruned_no_kd_model, val_loader, cls_device,
        temperature=cifar_kd_temperature, max_batches=8
    )
else:
    print("No no-KD pruned model found; run Cell 14 first to create `pruned_no_kd_model`.")

print("Teacher validation performance (CIFAR10):")
print(
    f"  batches={teacher_metrics['batches']} | samples={teacher_metrics['samples']} | "
    f"avg_forward_ms/batch={teacher_metrics['avg_forward_ms_per_batch']:.3f} | "
    f"avg_abs_logit={teacher_metrics['avg_abs_logit']:.6f} | "
    f"avg_logit_l2={teacher_metrics['avg_logit_l2']:.6f} | "
    f"top1_acc={teacher_metrics['top1_acc'] * 100.0:.2f}% ({teacher_metrics['acc_samples']} samples)"
)

print("Pruned student validation performance (CIFAR10):")
print(
    f"  batches={student_metrics['batches']} | samples={student_metrics['samples']} | "
    f"avg_forward_ms/batch={student_metrics['avg_forward_ms_per_batch']:.3f} | "
    f"avg_abs_logit={student_metrics['avg_abs_logit']:.6f} | "
    f"avg_logit_l2={student_metrics['avg_logit_l2']:.6f} | "
    f"top1_acc={student_metrics['top1_acc'] * 100.0:.2f}% ({student_metrics['acc_samples']} samples)"
)

if pruned_no_kd_metrics is not None:
    print("Pruned no-KD student validation performance (CIFAR10):")
    print(
        f"  batches={pruned_no_kd_metrics['batches']} | samples={pruned_no_kd_metrics['samples']} | "
        f"avg_forward_ms/batch={pruned_no_kd_metrics['avg_forward_ms_per_batch']:.3f} | "
        f"avg_abs_logit={pruned_no_kd_metrics['avg_abs_logit']:.6f} | "
        f"avg_logit_l2={pruned_no_kd_metrics['avg_logit_l2']:.6f} | "
        f"top1_acc={pruned_no_kd_metrics['top1_acc'] * 100.0:.2f}% ({pruned_no_kd_metrics['acc_samples']} samples)"
    )

print(f"Teacher validation CE loss: {teacher_val_loss:.6f}")
print(f"Student validation CE loss (with KD): {student_val_loss:.6f}")
if pruned_no_kd_val_loss is not None:
    print(f"Pruned model validation CE loss (without KD): {pruned_no_kd_val_loss:.6f}")
print(
    f"Validation KD loss (teacher vs student, T={cifar_kd_temperature}): "
    f"{val_kd_loss:.6f} over {loss_batches} batches"
)

Recorded 200 KD steps
First loss: 4.311590
Last loss:  2.492903
Teacher validation performance (CIFAR10):
  batches=8 | samples=128 | avg_forward_ms/batch=46.218 | avg_abs_logit=0.001000 | avg_logit_l2=0.009075 | top1_acc=0.00% (128 samples)
Pruned student validation performance (CIFAR10):
  batches=8 | samples=128 | avg_forward_ms/batch=45.952 | avg_abs_logit=0.939032 | avg_logit_l2=1.215596 | top1_acc=7.81% (128 samples)
Pruned no-KD student validation performance (CIFAR10):
  batches=8 | samples=128 | avg_forward_ms/batch=42.290 | avg_abs_logit=1.405778 | avg_logit_l2=1.794740 | top1_acc=0.00% (128 samples)
Teacher validation CE loss: 6.908537
Student validation CE loss (with KD): 3.479523
Pruned model validation CE loss (without KD): 9.601957
Validation KD loss (teacher vs student, T=2.0): 0.657622 over 8 batches
